# Chunking

* Chunking refers to the process of breaking down large documents(or text) into smaller, manageable pieces or chunks for more effective retrieval and generation.
* Chunk is small piece of text.
* Token is the chunk of text

## Chunking Strategies 
* **1) Fixed Size Chunking(sliding window) :**
* **2) Sentence-based chunking :**
* **3) Document based chunking :**
* **4) Semantic based chunking :**
* **5) Overlapping chunking :**
* **6) Recursive chunking :**
* **7) Agentic chunking :**
* **8) Token-Based chunking :**

### 1) Fixed Size Chunking :
* Fixed size chunking divides the text into fixed length pieces or windows.

In [21]:
from langchain_text_splitters import CharacterTextSplitter

text = " Fixed size chunking divides the text into fixed length pieces or windows. "
text_splitter = CharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

chunks = text_splitter.split_text(text)

### 2) Sentence-based Chunking : 
* It divides text into individual sentences , treating each as chunk. This is useful when working with structured text or datasets where each sentence contains meaningful information and needs to be processed independently.
* **Use cases :**
* Ideal for processing formal documents where individual sentences convey significant meaning. 

In [27]:
import nltk
from nltk.tokenize import sent_tokenize

# Download once
nltk.download('punkt')

text = """AI is transforming industries. It helps automate tasks. 
Companies are using AI for decision making. This improves efficiency."""

# Sentence-based chunks
chunks = sent_tokenize(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Chunk 1: AI is transforming industries.
Chunk 2: It helps automate tasks.
Chunk 3: Companies are using AI for decision making.
Chunk 4: This improves efficiency.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [33]:
from nltk.tokenize import sent_tokenize
from langchain_core.documents import Document

text = """AI is transforming industries. It helps automate tasks. 
Companies are using AI for decision making. This improves efficiency."""

sentences = sent_tokenize(text)

documents = [Document(page_content=sentence) for sentence in sentences]

for doc in documents:
    print(doc.page_content)

AI is transforming industries.
It helps automate tasks.
Companies are using AI for decision making.
This improves efficiency.


## 3) Document based Chunking :
* Documents based chunking involves treating each document of a text as a chunk.
* This approach is used when the corpus consist of multiple documents that need to be retrieved independently.

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = r"C:\Users\adity\DS_Projects\Langchain\data\campaign_PDF_Report.pdf"
loader = PyPDFLoader(file_path)

# 2. Load the data 
pages = loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

# Preview the results
print(f"Loaded {len(pages)} pages.")
print(f"Split into {len(chunks)} chunks.")
print(f"Content of first chunk:\n{chunks[0].page_content[:200]}...")

Loaded 3 pages.
Split into 8 chunks.
Content of first chunk:
Senior Data Scientist Assistant - Analysis Report
I. Exploratory Data Analysis
Campaign Data Summary
Dataset Shape: `34 rows × 12 features`
Duplicate Records: `0`
---
Features Overview
Numerical Featu...


## 4) Semantic-based Chunking(Dynamic chunking) :
* Semantic-based chunking involves splitting text into chunks based on the meaning of the content rather than fixed length or sentences boundaries.
* **Use Cases :**
* Use in situattions where meaningful semantic units.
* Useful for document summarization , knowledge extraction , and their advanced NLP tasks.

In [41]:
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
import numpy as np

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_chunking(text, similarity_threshold=0.7):
    sentences = sent_tokenize(text)
    embeddings = model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = np.dot(embeddings[i-1], embeddings[i]) / (
            np.linalg.norm(embeddings[i-1]) * np.linalg.norm(embeddings[i])
        )

        if sim >= similarity_threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]

    chunks.append(" ".join(current_chunk))
    return chunks


# Example
text = """AI is transforming industries. It helps automate tasks.
Machine learning improves predictions. Football is a popular sport.
Cricket is widely played in India."""

chunks = semantic_chunking(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")


Chunk 1: AI is transforming industries.
Chunk 2: It helps automate tasks.
Chunk 3: Machine learning improves predictions.
Chunk 4: Football is a popular sport.
Chunk 5: Cricket is widely played in India.


## 5) Overlaping Chunking :
* Overlaping chunking involves creating chunks that overlap with each other.
* **Ex :** One chunk might contain the last few words of the previous chunk.

In [48]:
from langchain_text_splitters import CharacterTextSplitter

text = """Overlapping chunking helps maintain context between chunks.
It ensures that important information is not lost when splitting text.
This is especially useful in RAG systems and LLM applications."""

text_splitter = CharacterTextSplitter(
    chunk_size=80,      # max characters per chunk
    chunk_overlap=20    # overlap between chunks
)

chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n")

Chunk 1:
Overlapping chunking helps maintain context between chunks.
It ensures that important information is not lost when splitting text.
This is especially useful in RAG systems and LLM applications.



## 6) Recursive Chunking:
* It divides the text into hirerchial , iterative manner using a set of separators.

In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=30,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n")

Chunk 1:
Overlapping chunking helps maintain context between chunks.

Chunk 2:
It ensures that important information is not lost when splitting text.

Chunk 3:
This is especially useful in RAG systems and LLM applications.



## 7) Agentic Chunking:
* Agentic Chunking leverages LLMs to determine how to chunk the text based on it's context.
* This is more dynamic approach where the model itself decides how much and what part of the text should from a chunk.

In [56]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.tokenize import sent_tokenize
import torch

# Load model 
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def agentic_chunking(text, max_chunk_sentences=3):
    sentences = sent_tokenize(text)
    chunks = []
    
    current_chunk = []

    for sentence in sentences:
        if not current_chunk:
            current_chunk.append(sentence)
            continue

        # Prompt to decide grouping
        prompt = f"""
        Decide if the following sentence belongs to the same topic as the chunk.

        Chunk: {" ".join(current_chunk)}
        Sentence: {sentence}

        Answer only YES or NO.
        """

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
        
        outputs = model.generate(**inputs, max_new_tokens=5)
        decision = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

        if "yes" in decision:
            current_chunk.append(sentence)
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]

        # optional safety limit
        if len(current_chunk) >= max_chunk_sentences:
            chunks.append(" ".join(current_chunk))
            current_chunk = []

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks



text = """AI is transforming industries. It automates tasks.
Machine learning improves predictions.
Football is a popular sport. Cricket is widely played in India."""

chunks = agentic_chunking(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Chunk 1: AI is transforming industries.
Chunk 2: It automates tasks.
Chunk 3: Machine learning improves predictions.
Chunk 4: Football is a popular sport.
Chunk 5: Cricket is widely played in India.
